DH演習2026 · 教材 第47版 · 2026-r47 · 2026-11-11

<details><summary>教材の版と相談用情報</summary>

相談時は次の情報を伝えてください。

```text
DH演習2026 · 教材 第47版
教材ID: 2026-r47
教材日付: 2026-11-11
活動: text_model
環境: Colab / Jupyter
ソース: ef540f48552065735e2d78a4fb68a88ddefed01a
```

</details>

[同じ版の教材・変更点](https://nlp.lang.osaka-u.ac.jp/dh/practicum/editions.html)


# 第4回：同じ発言をCSV・JSON・XMLで読み、何を一件とするか決める

上から順にセルを実行してください（Shift+Enter）。値を変えたら、そのセルから下を再実行します。

資料は芥川龍之介『二人小町』第1場の発言14・15（[TEI-EAJ aozora_tei](https://github.com/TEI-EAJ/aozora_tei/blob/77a675fc2771936f9544505d922d4cd45075338c/data/complete/tei_lib_lv3/86_tei.xml)、CC0）です。ルビの読みを除き、CSVとJSONは教材で作りました。

## 1. 同じ二つの発言を三つの形式で書く

`csv_text`は表、`json_text`は階層、`xml_text`はTEIの文書です。三つとも、黄泉の使の発言14と小野の小町の発言15を表します。発言15は、台詞・舞台指示・台詞の三つの区切りからなります。

実行前に予想してください。CSVの行数、JSONの`passages`の件数、XMLの`sp`の数は同じでしょうか。

In [ ]:
csv_text = '''passage,segment,kind,text
s1-sp14,1,text,いや、名前もあなたのように小町と云わなければいけないのです。
s1-sp15,1,text,小町！誰か小町と云う人はいなかったかしら。ああ、います。います。
s1-sp15,2,stage,（発作的に笑い出しながら）
s1-sp15,3,text,玉造の小町と云う人がいます。あの人を代りにつれて行って下さい。
'''

json_text = '''{"passages": [
  {"id": "s1-sp14", "speaker": "使", "who": "#messenger", "segments": [
    {"kind": "text", "text": "いや、名前もあなたのように小町と云わなければいけないのです。"}]},
  {"id": "s1-sp15", "speaker": "小町", "who": "#ono", "segments": [
    {"kind": "text", "text": "小町！誰か小町と云う人はいなかったかしら。ああ、います。います。"},
    {"kind": "stage", "text": "（発作的に笑い出しながら）"},
    {"kind": "text", "text": "玉造の小町と云う人がいます。あの人を代りにつれて行って下さい。"}]}
]}'''

xml_text = '''<div xmlns="http://www.tei-c.org/ns/1.0">
<sp who="#messenger"><speaker>使</speaker><p>いや、名前もあなたのように小町と云わなければいけないのです。</p></sp>
<sp who="#ono"><speaker>小町</speaker><p>小町！誰か小町と云う人はいなかったかしら。ああ、います。います。</p><stage>（発作的に笑い出しながら）</stage><p>玉造の小町と云う人がいます。あの人を代りにつれて行って下さい。</p></sp>
</div>'''

## 2. CSVを表として読む

`pd.read_csv()`はCSVを表にします。一行は一つの区切りです。`kind`を`"stage"`に変えると、舞台指示の行だけが残ります。

行数は発言の数ではありません。発言を数えるには、`passage`列の値が何種類あるかを数えます。

In [ ]:
import io

import pandas as pd

kind = "text"

table = pd.read_csv(io.StringIO(csv_text))
print("行数：", len(table))
print("発言の数：", table["passage"].nunique())
table[table["kind"] == kind]

## 3. JSONを階層として読む

`json.loads()`はJSONを辞書とリストにします。`model["passages"]`は発言のリストで、各発言の`segments`が区切りのリストです。発言者は発言ごとに一度だけ書いてあります。

In [ ]:
import json

model = json.loads(json_text)
print("発言の数：", len(model["passages"]))
for passage in model["passages"]:
    print(passage["id"], passage["speaker"], passage["who"], "区切り：", len(passage["segments"]))

## 4. XMLを文書として読む

`ET.fromstring()`はXMLを要素の木にします。`sp`が発言、`speaker`が発言者名、`p`が台詞、`stage`が舞台指示です。TEIの要素名には名前空間が付くので、`T`を前に付けて探します。

`tag`を`"stage"`に変えると、何を数えることになりますか。

In [ ]:
import xml.etree.ElementTree as ET

tag = "p"

T = "{http://www.tei-c.org/ns/1.0}"
div = ET.fromstring(xml_text)
print("発言の数：", len(div.findall(T + "sp")))
for sp in div.findall(T + "sp"):
    print(sp.get("who"), [e.text for e in sp.findall(T + tag)])

## 5. 階層を表に変換する

JSONの発言と区切りを順にたどると、CSVと同じ表を作れます。`columns`に`"speaker"`を足すと、発言者が区切りの行ごとに繰り返されます。

表に変換した後も、行数は区切りの数です。何を一件として数えるかは、形式ではなく問いで決まります。

In [ ]:
columns = ["passage", "segment", "kind", "text"]

rows = []
for passage in model["passages"]:
    for number, segment in enumerate(passage["segments"], 1):
        rows.append({"passage": passage["id"], "speaker": passage["speaker"], "segment": number, "kind": segment["kind"], "text": segment["text"]})
converted = pd.DataFrame(rows)[columns]
print("CSVの表と同じ：", converted.equals(table))
converted

## 確認問題

5節の`columns`に`"speaker"`を足して再実行してください。`speaker`が「小町」の行は何行ですか。小町の発言は何件ですか。`answer`に二つの数を入れてください。

In [ ]:
answer = {"rows": None, "speeches": None}

In [ ]:
#@title 答えを確かめる
if None in answer.values():
    print("結果：不合格（`answer`に二つの数を入れてください）")
elif answer["rows"] != 3:
    print("結果：不合格（5節の表で、`speaker`が「小町」の行を数えてください）")
elif answer["speeches"] == 3:
    print("結果：不合格（行は区切りです。発言は`passage`の値で数えてください）")
elif answer["speeches"] != 1:
    print("結果：不合格（3節で、発言者が「小町」の発言を数えてください）")
else:
    print("結果：合格")